In [2]:
!pip install minsearch --break-system-packages

Defaulting to user installation because normal site-packages is not writeable


In [7]:
import minsearch


In [8]:
import json

In [5]:
!wget https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json


--2025-10-03 19:15:32--  https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘documents.json.1’

documents.json.1        [     <=>            ] 858.41K   862KB/s    in 1.0s    

2025-10-03 19:15:34 (862 KB/s) - ‘documents.json.1’ saved [879011]



In [10]:
with open('documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)

In [11]:
documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [8]:
documents[0]


{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [12]:
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

In [13]:
q = 'the course has already started, can I still enroll?'


In [14]:
index.fit(documents)


In [ ]:
import os
from mistralai.client import MistralClient
api_key = "_YOUR_KEY"
client = MistralClient(api_key=api_key)

response = client.chat(
    model="open-mistral-7b",
    messages=[
        {"role": "user", "content":q}
    ]
)

print(response)

id='bd46b4740bf8416db4a3756c9309c618' object='chat.completion' created=1759574457 model='open-mistral-7b' choices=[ChatCompletionResponseChoice(index=0, message=ChatMessage(role='assistant', content="It depends on the specific course and the policy of the institution or platform where the course is offered. Some courses allow late enrollment, while others do not. It's best to contact the course provider directly to inquire about enrollment availability and any potential requirements or deadlines.", name=None, tool_calls=None, tool_call_id=None), finish_reason=<FinishReason.stop: 'stop'>)] usage=UsageInfo(prompt_tokens=15, total_tokens=73, completion_tokens=58)


In [1]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [21]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [4]:
def llm(prompt):
    response = client.chat(
    model="open-mistral-7b",
    messages=[
        {"role": "user", "content":prompt}
    ]
)
    return response.choices[0].message.content

In [22]:
query = 'how do I run kafka?'

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [23]:
rag(query)


'To run Kafka with Java, navigate to your project directory and run the following command:\n\n```\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```\n\nFor Python, create a virtual environment and install the required packages (assuming you\'re using a Linux or MacOS system):\n\n1. Create a virtual environment:\n\n```\npython -m venv env\nsource env/bin/activate\n```\n\n2. Install the dependencies using pip:\n\n```\npip install -r ../requirements.txt\n```\n\n3. Activate the virtual environment:\n\n```\nsource env/bin/activate\n```\n\n4. Deactivate the virtual environment when finished:\n\n```\ndeactivate\n```\n\nIf you encounter a "Permission denied" error on the "./build.sh" command when using Python, run the following command in the same directory:\n\n```\nchmod +x build.sh\n```\n\nRegarding the "ModuleNotFoundError: No module named \'kafka.vendor.six.moves\'", use the following alternative package:\n\n```\npip install kafka-python-n

In [24]:
rag('the course has already started, can I still enroll?')


"Based on the provided context, it appears that the course has already started on the 15th of January, 2024. However, the course materials will still be available for you to follow even if you didn't register before the start date. You can still submit the homework assignments, but keep in mind that there will be deadlines for turning in the final projects. It is recommended that you don't leave everything until the last minute.\n\nTo follow the course after it finishes, you can access all the materials and continue preparing for the next cohort. You can also start working on your final capstone project.\n\nBefore the course started, you were advised to prepare by installing and setting up all the dependencies and requirements such as a Google cloud account, Google Cloud SDK, Python 3 (installed with Anaconda), Terraform, and Git. You were also encouraged to look over the prerequisites and syllabus to see if you are comfortable with these subjects.\n\nIn terms of support, if you take t

In [25]:
documents[0]


{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [38]:
from elasticsearch import Elasticsearch


In [ ]:
es_client = Elasticsearch('http://localhost:9200', headers={"Accept": "application/vnd.elasticsearch+json; compatible-with=8"}
) 


In [41]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

index_name = "course-questions"
es_client.indices.create(index=index_name, body=index_settings)

BadRequestError: BadRequestError(400, 'media_type_header_exception', 'Invalid media-type value on headers [Accept, Content-Type]', Accept version must be either version 8 or 7, but found 9. Accept=application/vnd.elasticsearch+json; compatible-with=9)